# Ablation: Patch-Embed LN Removal vs. GITA Gamma Correction (SwinRCNN)

Reviewer comment: *"Layer normalization could potentially be replaced by the proposed gamma-correction-based input transformation."*

LN is per-sample (domain-agnostic), making it theoretically the best candidate for replacement by input-level gamma correction.  
We test **only `patch_embed.norm`** — the single LN between the input image and the first GITA anchor in layers.[01].

| Scenario | patch_embed.norm | layers.[01] LN (GITA anchors) | layers.[23]+ LN | TTA |
|----------|-----------------|------------------------------|-----------------|-----|
| **C** | Removed | Kept | Kept | None (direct test) |
| **D** | Removed | Kept (inside CascadeAnchor) | Kept | GITA (gamma ITM) |

If gamma (guided by layers.[01] LN anchors) can compensate for the missing patch-embed normalization, D should outperform C.

In [ ]:
import re
from os import path, makedirs, environ, system
import json

BATCH_SIZE     = 1
FIT_BATCH_SIZE = 40
TOTAL_ROUNDS   = 1
DATA_ROOT      = path.join(".", "data")
RESULT_ROOT    = path.join(".", "results-rebuttal")
DEVICE_NUM     = 0

In [ ]:
_ = system("nvidia-smi")

In [ ]:
import torch
import torch.nn as nn

environ["CUDA_VISIBLE_DEVICES"] = str(DEVICE_NUM)
environ["TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS"] = "1"
environ["TORCHDYNAMO_CAPTURE_DYNAMIC_OUTPUT_SHAPE_OPS"] = "1"

torch._dynamo.config.capture_scalar_outputs = True
torch._dynamo.config.suppress_errors = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"INFO: Using device - {device}:{DEVICE_NUM}")

In [ ]:
from ttadapters import datasets, models, methods
from ttadapters.datasets import scenarios
from ttadapters.utils.validator import DetectionEvaluator
from ttadapters.utils.visualizer import visualize_metrics
from ttadapters.methods.auto import CONFIG_MAPPING

import pandas as pd
pd.options.display.float_format = lambda x: f"{x*100 if x < 1 else x:.4f}"

In [ ]:
datasets.patch_fast_download_for_object_detection()

train_dataset = datasets.SHIFTContinuousSubsetForObjectDetection(root=DATA_ROOT, train=True)
CLASSES = train_dataset.classes
print(f"INFO: Number of classes - {len(CLASSES)} {CLASSES}")

In [ ]:
def remove_layer_norm(model: nn.Module, exclude_pattern: str = None, include_pattern: str = None) -> int:
    """Replace LayerNorm layers with nn.Identity.

    Args:
        exclude_pattern: regex. Matching layer names are KEPT.
        include_pattern: regex. If provided, ONLY matching layers are removed.
                         exclude_pattern still applies as a safety guard.

    Returns:
        Number of replaced layers.
    """
    exc_re = re.compile(exclude_pattern) if exclude_pattern else None
    inc_re = re.compile(include_pattern) if include_pattern else None
    to_replace = []

    for name, module in model.named_modules():
        is_ln = isinstance(module, nn.LayerNorm) or "LayerNorm" in module.__class__.__name__
        if is_ln:
            if inc_re:
                if inc_re.search(name) and (exc_re is None or not exc_re.search(name)):
                    to_replace.append(name)
            elif exc_re is None or not exc_re.search(name):
                to_replace.append(name)

    for name in to_replace:
        parts = name.split(".")
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], nn.Identity())

    return len(to_replace)


def make_json_serializable(obj):
    if isinstance(obj, list):
        return [make_json_serializable(i) for i in obj]
    if isinstance(obj, dict):
        return {(k.value if hasattr(k, "value") else k): make_json_serializable(v) for k, v in obj.items()}
    return obj


result_dir = path.join(RESULT_ROOT, "swinrcnn", "continual_tta", "patch_embed_ln_removal_ablation")
makedirs(result_dir, exist_ok=True)

---
## Scenario C — patch_embed.norm Removed, Direct Test

Remove only the patch-embed LayerNorm (between input and first layers.[01] anchor) and run inference without adaptation.
Baseline: how much does removing the patch-embed normalization degrade mAP?

In [ ]:
base_model_c, load_result_c = models.SwinRCNNForObjectDetection.from_dataset(dataset=datasets.SHIFTDataset)
data_preparation = base_model_c.DataPreparation(train_dataset, evaluation_mode=True)
print("INFO: Model state loaded -", load_result_c)

# Remove ONLY patch_embed.norm — the LN between the input image and the first layers.[01] anchor.
n_c = remove_layer_norm(base_model_c, include_pattern=r"patch_embed")
print(f"C: Replaced {n_c} LayerNorm -> Identity (patch_embed only; layers.[01]+ kept)")

base_model_c.to(device)
base_model_c.eval()

scenario_params = dict(root=DATA_ROOT, valid=True, transforms=data_preparation.transforms, order=None)
loader_params   = dict(batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_preparation.collate_fn)

In [ ]:
tta_c = methods.MethodContainer(**{"C_SwinRCNN_PatchEmbedLNRemoved": base_model_c})

evaluator_c = DetectionEvaluator(
    tta_c.methods(), classes=CLASSES,
    data_preparation=data_preparation,
    dtype=torch.float32, device=device,
    no_grad=True
)

result_c = []
for rd, this in tta_c.go_rounds(end_round=TOTAL_ROUNDS):
    continual_scenario_c = scenarios.SHIFTDiscreteScenarioForContinualTTA(**scenario_params)
    result = visualize_metrics(continual_scenario_c(**loader_params).play(evaluator_c, index=this))
    result_c.append(result)
    with open(path.join(result_dir, f"C_result_r{rd}_b{BATCH_SIZE}.json"), "w", encoding="utf-8") as f:
        json.dump(make_json_serializable(result), f)

print(f"\nC done. Saved to: {result_dir}")

---
## Scenario D — patch_embed.norm Removed + GITA

Apply GITA first so that layers.[01] block norms are wrapped with CascadeAnchor, then remove only patch_embed.norm.
The layers.[01] anchors observe pre-norm activations; if gamma correction compensates for the missing
patch-embed normalization, the downstream features should realign to the source distribution.

**Removed**: `patch_embed.norm` only
**Kept (inside CascadeAnchor)**: `layers.[01].blocks.*.norm[12]` — LN still runs, providing anchor signal for gamma
**Kept**: `layers.[23]` and all other LN — the rest of the transformer is intact

`fit()` is required: unlike BN, LN has no running stats — source statistics are computed from training data.

In [ ]:
base_model_d, load_result_d = models.SwinRCNNForObjectDetection.from_dataset(dataset=datasets.SHIFTDataset)
print("INFO: Model state loaded -", load_result_d)

In [ ]:
config_d = CONFIG_MAPPING["gita_engine"].from_preset(base_model_d)
adaptive_model_d = methods.AutoAdaptationEngineForObjectDetection.from_config(config_d, base_model=base_model_d)
adaptive_model_d.to(device)

# Remove ONLY patch_embed.norm — the LN between the ITM (gamma) and the first layers.[01] anchor.
# Exclude *.norm*.norm paths to preserve LN inside CascadeAnchor (e.g. norm1.norm, norm2.norm).
EXCLUDE_ANCHOR_INTERNALS = r"\.norm\w*\.norm$"

n_d = remove_layer_norm(adaptive_model_d.base_model, include_pattern=r"patch_embed", exclude_pattern=EXCLUDE_ANCHOR_INTERNALS)
print(f"D: Replaced {n_d} patch_embed LayerNorm -> Identity (layers.[01] CascadeAnchor internals preserved)")

In [ ]:
adaptive_model_d.fit(data_preparation, batch_size=FIT_BATCH_SIZE, shuffle=False)

In [ ]:
adaptive_model_d.online()

tta_d = methods.MethodContainer(**{
    f"D_SwinRCNN_PatchEmbedLNRemoved_{adaptive_model_d.model_type}": adaptive_model_d
})

evaluator_d = DetectionEvaluator(
    tta_d.methods(), classes=CLASSES,
    data_preparation=data_preparation,
    dtype=torch.float32, device=device,
    no_grad=False
)

result_d = []
for rd, this in tta_d.go_rounds(end_round=TOTAL_ROUNDS):
    continual_scenario_d = scenarios.SHIFTDiscreteScenarioForContinualTTA(**scenario_params)
    result = visualize_metrics(continual_scenario_d(**loader_params).play(evaluator_d, index=this))
    result_d.append(result)
    with open(path.join(result_dir, f"D_result_r{rd}_b{BATCH_SIZE}.json"), "w", encoding="utf-8") as f:
        json.dump(make_json_serializable(result), f)

print(f"\nD done. Saved to: {result_dir}")